# Generación reproducible del dataset aumentado — Primer entregable

Este notebook documenta la preparación de la base de datos del curso de precálculo y la generación reproducible de los conjuntos de entrenamiento y prueba que serán utilizados en las etapas posteriores del proyecto.

## Fuente original y formato estandarizado

La información original se encuentra en `Data/Datos_Brutos.csv`. Este archivo contiene 1.527 registros, 31 columnas y la información detallada del curso, incluyendo los resultados de ALEKS, los datos académicos, la asignatura relacionada y las calificaciones. La fuente cubre los periodos `202310`, `202330`, `202410`, `202430`, `202510` y `202530`.

El formato de salida se toma de `cabeceras.csv`. Este archivo define el esquema estandarizado de 17 columnas y los nombres correctos de las variables de resultado:

- En la fuente original, la calificación del primer parcial se llama `nota primer parcial`.
- En el esquema de modelado, la variable objetivo se llama `nota_primer_parcial`.
- En la fuente original, la calificación final se llama `nota final`.
- En el esquema de modelado, la columna correspondiente se llama `nota_final`.

La fuente original contiene 1.497 registros con valor disponible en la variable objetivo y 30 registros sin esa calificación. Debido a que 1.527 observaciones no alcanzan el mínimo de 20.000 registros solicitado, se construye un conjunto aumentado con una meta de 30.000 filas: 24.000 para entrenamiento y 6.000 para prueba.

## Propósito del procedimiento

El proceso transforma los nombres y el orden de las columnas de la fuente original al esquema estandarizado, descarta las filas que no tienen objetivo para el aprendizaje supervisado, separa los estudiantes en entrenamiento y prueba, y genera variantes sintéticas dentro de cada partición. Las perturbaciones se mantienen dentro de límites plausibles para conservar la estructura académica de los datos.

La variable objetivo es `nota_primer_parcial`. La columna `nota_final` se conserva en las salidas por razones de trazabilidad, pero no debe utilizarse como predictor del primer parcial.

## Resumen del flujo

1. Cargar la fuente original y el archivo que define el formato de salida.
2. Estandarizar las columnas mediante sus nombres, sin depender de posiciones fijas.
3. Eliminar las filas sin `nota_primer_parcial` antes de crear grupos o particiones.
4. Agrupar los registros por estudiante para evitar que un mismo estudiante aparezca en entrenamiento y prueba.
5. Dividir de forma estratificada usando bins construidos a partir del promedio del objetivo por estudiante.
6. Aumentar entrenamiento y prueba de manera independiente.
7. Escribir los dos archivos con separador punto y coma, decimal con coma y codificación UTF-8 con BOM.
8. Validar tamaños, encabezados, rangos, duplicados y ausencia de solapamiento entre particiones.

## 0. Configuración

Esta sección define las rutas, la semilla de aleatoriedad, el tamaño de las particiones y las variables que controlan el proceso reproducible.

`RAW_SOURCE` apunta a la fuente original del curso, `Data/Datos_Brutos.csv`. `HEADER_SOURCE` apunta al archivo `cabeceras.csv`, que se utiliza como referencia para conservar los nombres estandarizados y el orden de las 17 columnas de salida.

La meta se establece en 30.000 registros, distribuidos en 24.000 observaciones de entrenamiento y 6.000 de prueba. Esta configuración supera el mínimo de 20.000 registros y permite evaluar posteriormente los modelos sobre una partición independiente.

In [2]:
from collections import defaultdict
from pathlib import Path
import csv
import math
import random
import statistics

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path(".")
RAW_SOURCE = ROOT / "Data" / "Datos_Brutos.csv"
HEADER_SOURCE = ROOT / "cabeceras.csv"
TRAIN_OUTPUT = ROOT / "Datos_Aleks_Augmented_30000_r_2_train.csv"
TEST_OUTPUT = ROOT / "Datos_Aleks_Augmented_30000_r_2_test.csv"

SEED = 20260921
TRAIN_ROWS = 24000
TEST_ROWS = 6000
TARGET = "nota_primer_parcial"
FINAL_GRADE = "nota_final"

print(f"Fuente original: {RAW_SOURCE}")
print(f"Cabeceras de salida: {HEADER_SOURCE}")
print(f"Semilla: {SEED}")


Fuente original: Data\Datos_Brutos.csv
Cabeceras de salida: cabeceras.csv
Semilla: 20260921


## 1. Carga, estandarización y preparación del objetivo

Se cargan dos documentos con funciones diferentes:

- `Data/Datos_Brutos.csv` contiene la información original del curso, con 31 columnas y nombres provenientes de la fuente académica.
- `cabeceras.csv` define el formato estandarizado que utilizarán las etapas de análisis y modelado, con 17 columnas.

La correspondencia entre los nombres de las calificaciones es explícita: `nota primer parcial` se transforma en `nota_primer_parcial`, y `nota final` se transforma en `nota_final`. La selección de columnas se realiza mediante nombres de cabecera para que la transformación sea trazable y no dependa de posiciones fijas.

Las filas sin `nota_primer_parcial` se identifican y se excluyen antes de calcular grupos, bins o particiones. De esta forma, todas las filas que llegan a los archivos supervisados tienen un valor válido para el objetivo.

In [3]:
def parse_number(value):
    text = (value or "").strip()
    if not text or text == "-":
        return None
    return float(text.replace(",", "."))

def clip(value, low, high):
    return max(low, min(high, value))

def decimal_places(value):
    text = (value or "").strip()
    if "," not in text:
        return 0
    tail = text.split(",", 1)[1]
    return len(tail) if tail.replace("-", "").isdigit() else 0

def format_number(value, col, integer_columns, precision):
    if value is None:
        return ""
    if col in integer_columns:
        return str(int(round(value)))
    text = f"{value:.{precision.get(col, 6)}f}".rstrip("0").rstrip(".")
    if text in ("", "-0"):
        text = "0"
    return text.replace(".", ",")

with RAW_SOURCE.open("r", encoding="utf-8-sig", newline="") as handle:
    raw_reader = csv.reader(handle, delimiter=";")
    raw_header = next(raw_reader)
    raw_rows_all = [[value.strip() for value in row] for row in raw_reader]

with HEADER_SOURCE.open("r", encoding="utf-8-sig", newline="") as handle:
    header_reader = csv.reader(handle, delimiter=";")
    output_header = next(header_reader)

raw_target_index = raw_header.index("nota primer parcial")
print(f"Registros en la fuente original: {len(raw_rows_all):,}")
print(f"Registros con objetivo disponible: {sum(bool(row[raw_target_index].strip()) for row in raw_rows_all):,}")
print(f"Registros sin objetivo: {sum(not row[raw_target_index].strip() for row in raw_rows_all):,}")
print(f"Columnas de la fuente original: {len(raw_header)}")
print(f"Columnas del formato estandarizado: {len(output_header)}")
print(f"Variable objetivo estandarizada: {TARGET}")
print(f"Columna de resultado conservada: {FINAL_GRADE}")

if len(output_header) != 17:
    raise ValueError(f"El formato de salida debe tener 17 columnas, no {len(output_header)}")
if TARGET not in output_header or FINAL_GRADE not in output_header:
    raise ValueError("Faltan las columnas de notas en el formato de salida")

raw_idx = {name: index for index, name in enumerate(raw_header)}
target_raw_index = raw_idx["nota primer parcial"]
final_raw_index = raw_idx["nota final"]
pidm_raw_index = raw_idx["pidm en matricula"]

rows_without_target = [
    row for row in raw_rows_all
    if parse_number(row[target_raw_index]) is None
]
raw_rows = [
    row for row in raw_rows_all
    if parse_number(row[target_raw_index]) is not None
]

if not raw_rows:
    raise ValueError("No quedaron filas con nota_primer_parcial válida")

print(f"Filas originales: {len(raw_rows_all):,}")
print(f"Filas eliminadas por target vacío: {len(rows_without_target):,}")
print(f"Filas válidas antes de dividir: {len(raw_rows):,}")


Registros en la fuente original: 1,527
Registros con objetivo disponible: 1,497
Registros sin objetivo: 30
Columnas de la fuente original: 31
Columnas del formato estandarizado: 17
Variable objetivo estandarizada: nota_primer_parcial
Columna de resultado conservada: nota_final
Filas originales: 1,527
Filas eliminadas por target vacío: 30
Filas válidas antes de dividir: 1,497


## 2. Conversión al esquema estandarizado y división estratificada

La información original se transforma al esquema de 17 columnas definido por `cabeceras.csv`. Los identificadores personales y los campos auxiliares de la fuente original no se incorporan al archivo final; el identificador interno del estudiante se utiliza temporalmente para realizar una división sin contaminación entre entrenamiento y prueba.

La estratificación se construye a partir del promedio de `nota_primer_parcial` por estudiante. Los bins permiten conservar una composición comparable de niveles de desempeño en ambas particiones. Primero se asignan los estudiantes a entrenamiento y prueba; después se realiza la aumentación dentro de cada partición.

In [4]:
mapping = [
    raw_idx["Icfes nuevo"],
    raw_idx["Icfes Matemáticas"],
    raw_idx["Tiempo empleado en la verificación de conocimientos"],
    raw_idx["Dominados (cantidad de temas)"],
    raw_idx["Categoria temas inicial"],
    raw_idx["avance precalculo aleks"],
    raw_idx["Progreso (cantidad de temas)"],
    raw_idx["Categoria temas final"],
    raw_idx["Tiempo Total Tiempo"],
    raw_idx["Tiempo total - Aprendidos/hora"],
    raw_idx["programa en matricula"],
    raw_idx["division en matricula"],
    raw_idx["Mat Curso Asignatura Relacionada"],
    raw_idx["avance cuartiles"],
    raw_idx["cumplimiento temas"],
    target_raw_index,
    final_raw_index,
]

mapped_rows = [[row[index] for index in mapping] for row in raw_rows]
target_col = output_header.index(TARGET)
final_col = output_header.index(FINAL_GRADE)

# Precisión y límites derivados de la fuente anonimizada ya mapeada.
numeric_columns = {0, 1, 2, 3, 5, 6, 8, 9, 15, 16}
integer_columns = {0, 1, 3, 5, 6}
precision = {
    col: max(
        decimal_places(row[col])
        for row in mapped_rows
        if row[col].strip()
    )
    for col in numeric_columns
}
source_numeric_values = {
    col: [
        parse_number(row[col])
        for row in mapped_rows
        if parse_number(row[col]) is not None
    ]
    for col in numeric_columns
}
source_limits = {
    col: (min(values), max(values)) if values else (0.0, 0.0)
    for col, values in source_numeric_values.items()
}

groups = defaultdict(list)
for index, row in enumerate(raw_rows):
    groups[row[pidm_raw_index]].append(index)

group_score = {}
for key, indices in groups.items():
    values = [parse_number(raw_rows[index][target_raw_index]) for index in indices]
    group_score[key] = statistics.fmean(values)

group_keys = list(groups)
scores = pd.Series([group_score[key] for key in group_keys], index=group_keys)
bins = pd.qcut(scores.rank(method="first"), q=10, labels=False)
bin_by_group = dict(zip(group_keys, bins.astype(int)))

train_groups, test_groups = train_test_split(
    group_keys,
    test_size=0.2,
    random_state=SEED,
    stratify=bins,
)
train_groups = set(train_groups)
test_groups = set(test_groups)

train_source = [
    mapped_rows[index]
    for key in train_groups
    for index in groups[key]
]
test_source = [
    mapped_rows[index]
    for key in test_groups
    for index in groups[key]
]

train_bin_counts = pd.Series([bin_by_group[key] for key in train_groups]).value_counts().sort_index()
test_bin_counts = pd.Series([bin_by_group[key] for key in test_groups]).value_counts().sort_index()
bin_summary = pd.DataFrame({
    "train_grupos": train_bin_counts,
    "test_grupos": test_bin_counts,
}).fillna(0).astype(int)

print(f"Grupos train antes de aumentar: {len(train_groups):,}")
print(f"Grupos test antes de aumentar: {len(test_groups):,}")
print(f"Filas train antes de aumentar: {len(train_source):,}")
print(f"Filas test antes de aumentar: {len(test_source):,}")
print("Representación de bins por grupo:")
display(bin_summary)

if train_groups & test_groups:
    raise AssertionError("Un grupo de estudiante apareció en train y test")


Grupos train antes de aumentar: 1,184
Grupos test antes de aumentar: 297
Filas train antes de aumentar: 1,196
Filas test antes de aumentar: 301
Representación de bins por grupo:


,train_grupos,test_grupos
0,119,30
1,118,30
2,118,30
3,118,30
4,119,29
5,118,30
6,118,30
7,118,30
8,119,29
9,119,29


## 3. Aumentación independiente de cada partición

Las filas observadas de cada partición se conservan y las nuevas observaciones se generan únicamente a partir de la partición correspondiente. Este orden evita que una variante sintética derivada de un estudiante del conjunto de entrenamiento termine en el conjunto de prueba.

Las perturbaciones se aplican de manera controlada sobre variables numéricas y calificaciones. Se respetan los límites observados y la relación general entre las variables académicas, con el propósito de ampliar la cantidad de registros sin modificar de forma arbitraria la estructura del problema.

El resultado debe interpretarse como un conjunto aumentado para fines de modelado y experimentación, no como una sustitución de la información académica original ni como evidencia de nuevos estudiantes observados.

In [5]:
def make_variant(base, rng):
    row = list(base)
    z_total = rng.gauss(0.0, 1.0)
    z_math = rng.gauss(0.0, 1.0)

    # Icfes con perturbación correlacionada.
    total = parse_number(row[0])
    math_score = parse_number(row[1])
    rho_icfes = 0.8126
    rho_icfes_complement = math.sqrt(1.0 - rho_icfes * rho_icfes)
    if total is not None:
        value = clip(total + max(1.0, 0.05 * 47.7742) * z_total, 0.0, 500.0)
        row[0] = format_number(round(value), 0, integer_columns, precision)
    if math_score is not None:
        noise = 0.05 * 10.1058 * (
            rho_icfes * z_total + rho_icfes_complement * z_math
        )
        value = clip(math_score + noise, 0.0, 100.0)
        row[1] = format_number(round(value), 1, integer_columns, precision)

    # Tiempo de verificación.
    verification_time = parse_number(row[2])
    if verification_time is not None:
        value = verification_time * math.exp(rng.gauss(0.0, 0.06))
        row[2] = format_number(
            clip(value, 0.0, source_limits[2][1]),
            2,
            integer_columns,
            precision,
        )

    # Dominio inicial, limitado por el dominio final.
    initial_domain = parse_number(row[3])
    final_domain = parse_number(row[6])
    if initial_domain is not None and final_domain is not None:
        sigma = max(
            0.5,
            min(2.0, 0.7 + 0.025 * math.sqrt(max(final_domain, 0.0))),
        )
        value = round(initial_domain + rng.gauss(0.0, sigma))
        row[3] = format_number(
            clip(value, 0.0, final_domain),
            3,
            integer_columns,
            precision,
        )

    # Tiempo total y aprendidos por hora.
    total_time = parse_number(row[8])
    learned_per_hour = parse_number(row[9])
    time_factor = None
    if total_time is not None:
        time_factor = math.exp(rng.gauss(0.0, 0.05))
        row[8] = format_number(
            clip(total_time * time_factor, 0.0, source_limits[8][1]),
            8,
            integer_columns,
            precision,
        )
    if learned_per_hour is not None and time_factor is not None:
        noise = rng.gauss(0.0, max(0.005, 0.01 * learned_per_hour))
        value = max(0.0, learned_per_hour / time_factor + noise)
        row[9] = format_number(
            clip(value, 0.0, source_limits[9][1]),
            9,
            integer_columns,
            precision,
        )

    # Ambas notas reciben una variación pequeña y correlacionada.
    grade_shared = rng.gauss(0.0, 1.0)
    grade_specific = rng.gauss(0.0, 1.0)
    rho_grades = 0.82
    rho_grades_complement = math.sqrt(1.0 - rho_grades * rho_grades)

    first_grade = parse_number(row[15])
    if first_grade is not None:
        value = clip(first_grade + 0.08 * grade_shared, 0.5, 5.0)
        row[15] = format_number(value, 15, integer_columns, precision)

    final_grade = parse_number(row[16])
    if final_grade is not None:
        correlated_noise = (
            rho_grades * grade_shared
            + rho_grades_complement * grade_specific
        )
        value = clip(final_grade + 0.08 * correlated_noise, 0.5, 5.0)
        row[16] = format_number(value, 16, integer_columns, precision)

    return tuple(row)


def augment_partition(source_rows, target_rows, seed, seen_before):
    if len(source_rows) > target_rows:
        raise ValueError("La partición fuente supera su tamaño objetivo")

    if any(tuple(row) in seen_before for row in source_rows):
        raise ValueError("Existe solapamiento exacto entre las particiones fuente")

    rng = random.Random(seed)
    generated = list(source_rows)
    seen = set(seen_before)
    seen.update(tuple(row) for row in source_rows)
    retries = 0

    while len(generated) < target_rows:
        candidate = make_variant(source_rows[rng.randrange(len(source_rows))], rng)
        while candidate in seen:
            retries += 1
            candidate = make_variant(source_rows[rng.randrange(len(source_rows))], rng)
            if retries > 100000:
                raise RuntimeError("No se pudo evitar un duplicado exacto")
        seen.add(candidate)
        generated.append(list(candidate))

    return generated, retries, seen


## 4. Generación y escritura de los dos archivos

En esta etapa se generan las metas establecidas para el proyecto:

- `Datos_Aleks_Augmented_30000_r_2_train.csv`: 24.000 registros para entrenamiento.
- `Datos_Aleks_Augmented_30000_r_2_test.csv`: 6.000 registros para prueba.

Ambos archivos conservan el encabezado estandarizado, el separador punto y coma, los decimales con coma y la codificación UTF-8 con BOM. El uso de una semilla fija permite reproducir la misma preparación cuando se ejecuta nuevamente el notebook.

In [6]:
train_rows, train_retries, seen_train = augment_partition(
    train_source,
    TRAIN_ROWS,
    SEED + 10,
    set(),
)
test_rows, test_retries, seen_all = augment_partition(
    test_source,
    TEST_ROWS,
    SEED + 20,
    seen_train,
)

for output_path, rows in (
    (TRAIN_OUTPUT, train_rows),
    (TEST_OUTPUT, test_rows),
):
    with output_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.writer(handle, delimiter=";", lineterminator="\n")
        writer.writerow(output_header)
        writer.writerows(rows)

print(f"Train generado: {TRAIN_OUTPUT} -> {len(train_rows):,} filas")
print(f"Test generado: {TEST_OUTPUT} -> {len(test_rows):,} filas")
print(f"Reintentos para evitar duplicados -> train: {train_retries}, test: {test_retries}")


Train generado: Datos_Aleks_Augmented_30000_r_2_train.csv -> 24,000 filas
Test generado: Datos_Aleks_Augmented_30000_r_2_test.csv -> 6,000 filas
Reintentos para evitar duplicados -> train: 2, test: 1


## 5. Validaciones finales

Las validaciones comprueban que los archivos generados cumplan las condiciones necesarias para continuar con el análisis exploratorio y el modelado:

- El encabezado de entrenamiento y prueba coincide con el formato estandarizado.
- Las particiones tienen exactamente 24.000 y 6.000 filas.
- Todas las filas tienen las 17 columnas esperadas.
- `nota_primer_parcial` y `nota_final` se encuentran dentro del rango de calificación definido.
- No existen valores vacíos en la variable objetivo.
- No existen duplicados completos dentro de cada partición.
- No existe solapamiento exacto entre entrenamiento y prueba.

In [7]:
def load_output(path):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle, delimiter=";")
        return next(reader), list(reader)

train_header, train_rows_check = load_output(TRAIN_OUTPUT)
test_header, test_rows_check = load_output(TEST_OUTPUT)
train_df = pd.read_csv(TRAIN_OUTPUT, sep=";", decimal=",", encoding="utf-8-sig")
test_df = pd.read_csv(TEST_OUTPUT, sep=";", decimal=",", encoding="utf-8-sig")

assert train_header == output_header
assert test_header == output_header
assert len(train_rows_check) == TRAIN_ROWS
assert len(test_rows_check) == TEST_ROWS
assert all(len(row) == len(output_header) for row in train_rows_check)
assert all(len(row) == len(output_header) for row in test_rows_check)
assert train_df[TARGET].notna().all()
assert test_df[TARGET].notna().all()
assert train_df[TARGET].between(0.5, 5.0).all()
assert test_df[TARGET].between(0.5, 5.0).all()
assert train_df[FINAL_GRADE].between(0.5, 5.0).all()
assert test_df[FINAL_GRADE].between(0.5, 5.0).all()
assert len(set(map(tuple, train_rows_check)) & set(map(tuple, test_rows_check))) == 0
assert train_df.duplicated().sum() == 0
assert test_df.duplicated().sum() == 0

print("Validaciones aprobadas")
print(f"Target vacío en train: {int(train_df[TARGET].isna().sum())}")
print(f"Target vacío en test: {int(test_df[TARGET].isna().sum())}")
print(f"Media nota parcial -> train: {train_df[TARGET].mean():.4f} | test: {test_df[TARGET].mean():.4f}")
print(f"Media nota final -> train: {train_df[FINAL_GRADE].mean():.4f} | test: {test_df[FINAL_GRADE].mean():.4f}")


Validaciones aprobadas
Target vacío en train: 0
Target vacío en test: 0
Media nota parcial -> train: 3.5317 | test: 3.5310
Media nota final -> train: 3.4177 | test: 3.3484


## Resultado

La fuente original se transforma al esquema estandarizado antes de generar las variantes sintéticas. Las filas sin `nota_primer_parcial` no llegan a los archivos supervisados, y la división por estudiante se realiza antes de la aumentación para evitar contaminación entre entrenamiento y prueba.

Los dos archivos resultantes constituyen la base reproducible para el análisis exploratorio, la selección de variables y la implementación del modelo base.